# 26. Gói artefact và bản trích số liệu

Sổ tay cuối của đợt hợp nhất không tính thêm điều gì mới. Nó làm ba việc: kiểm tra đầy đủ bộ artefact, sinh bản trích số liệu dạng văn bản để dán trực tiếp vào bản soạn thảo, và nén toàn bộ thành một tệp duy nhất để tải về.

Bản trích số liệu là artefact quan trọng nhất của đợt này. Mọi sai sót số liệu đã từng xảy ra trong đề tài đều phát sinh ở khâu chép tay từ màn hình kết quả sang bản báo cáo. Khi có bản trích sinh tự động từ đúng tệp hợp nhất, khâu chép tay bị loại bỏ.

Mỗi tệp trong gói đều có mã băm, nên khi bản báo cáo đã nộp còn có thể đối chiếu về đúng phiên bản artefact đã dùng để viết nó.

Không huấn luyện mô hình. Chi phí dưới một phút.

> **Vị trí trong chuỗi tái lập:** notebook 26 trên tổng số 31. Thứ tự chạy và sơ đồ phụ thuộc nằm trong `README.md`.
>
> **Cần chạy trước:** 03-Tien_xu_ly_va_thuc_nghiem_chinh, 09-Ba_ham_mat_mat_hai_neo, 10-Luoi_nam_hat_giong, 11-Chi_so_theo_nhom_kenh, 12-Phan_tang_theo_do_tre, 13-Moc_trung_binh_dich_chuyen, 14-Doi_chung_moc_manh_nhat, 15-Neo_B_ham_tuyet_doi_nam_hat_giong, 16-Kiem_tra_thang_do_kenh, 17-Neo_theo_nguong_do_phu, 18-Phan_tang_gop_tang_cuoi, 22-Hop_nhat_chi_so, 23-Kiem_dinh_dong_nhat, 24-Bang_kenh_va_phan_tang, 25-Bo_hinh_chinh.
>
> **Thư mục kết quả:** `Output/processed/w3_artefacts_tong_hop/`.
>
> **Mã định danh nội bộ:** `W3.W4_30`. Mã này còn xuất hiện trong tên khóa JSON, tên hình và nhật ký chạy; nó được giữ nguyên để không phá phép đối chiếu chéo và mã băm của bản khai báo trước.

## 1. Tham số và đường dẫn

In [1]:
#@title Tham số chạy { display-mode: "form" }
PROJECT_DIR = "/content/drive/MyDrive/AIO2026_Conquer_M4_DLinear_NLinear_Slot4"  #@param {type:"string"}
READ_TAG = "chot"  #@param {type:"string"}
RUN_TAG = "w3_artefacts_tong_hop"  #@param {type:"string"}
TEN_GOI = "W3_artefacts_bao_cao"  #@param {type:"string"}
TAI_VE_SAU_KHI_NEN = True  #@param {type:"boolean"}
SEQ_LEN = 36     #@param {type:"integer"}
PRED_LEN = 12    #@param {type:"integer"}
N_SEEDS = 3      #@param {type:"integer"}
EPOCHS = 60      #@param {type:"integer"}
PATIENCE = 10    #@param {type:"integer"}
BATCH = 128      #@param {type:"integer"}
LR = 1e-3        #@param {type:"number"}

import os, sys, json, time, random, datetime, platform, importlib, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
else:
    PROJECT_DIR = os.environ.get("P12_PROJECT", "./AIO2026_Conquer_M4_DLinear_NLinear_Slot4")

PROJECT = Path(PROJECT_DIR)
OUT = PROJECT / "Output"
# Đọc dữ liệu đã tiền xử lý của lần chạy chốt, ghi kết quả sang một thư mục con riêng.
PROC = OUT / "processed" / READ_TAG
RUN_DIR = OUT / "processed" / RUN_TAG
RUNS = RUN_DIR / "runs"
FIGS = RUN_DIR / "figures"
LOGS = RUN_DIR / "logs"
for d in (RUN_DIR, RUNS, FIGS, LOGS):
    d.mkdir(parents=True, exist_ok=True)

assert (PROC / "meta.json").exists(), (
    "Không thấy dữ liệu đã tiền xử lý trong " + str(PROC) +
    ". Hãy chạy notebook 03-Tien_xu_ly_va_thuc_nghiem_chinh trước.")
meta = json.loads((PROC / "meta.json").read_text())
channels = meta["channels"]
EPS = 1e-8

print("Thư mục đọc :", PROC)
print("Thư mục ghi :", RUN_DIR)
print("Số kênh     :", len(channels), "| ngưỡng cắt sigma của lần chạy nguồn:", meta["clip_sigma"])


Mounted at /content/drive
Thư mục đọc : /content/drive/MyDrive/AIO2026_Conquer_M4_DLinear_NLinear_Slot4/Output/processed/chot
Thư mục ghi : /content/drive/MyDrive/AIO2026_Conquer_M4_DLinear_NLinear_Slot4/Output/processed/w3_artefacts_tong_hop
Số kênh     : 37 | ngưỡng cắt sigma của lần chạy nguồn: 0.0


In [2]:
REQUIRED = {"numpy": "1.24.0", "torch": "2.0.0", "matplotlib": "3.7.0", "scipy": "1.10.0"}


def ver_tuple(v):
    out = []
    for part in str(v).split(".")[:3]:
        digits = "".join(ch for ch in part if ch.isdigit())
        out.append(int(digits) if digits else 0)
    return tuple(out + [0] * (3 - len(out)))


missing = []
for name, floor in REQUIRED.items():
    try:
        mod = importlib.import_module(name)
        if ver_tuple(mod.__version__) < ver_tuple(floor):
            missing.append(name + ">=" + floor)
    except ImportError:
        missing.append(name + ">=" + floor)

if missing:
    print("Cài bổ sung:", " ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=True)

import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
import scipy
from scipy import stats

ENV = {
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    "run_tag": RUN_TAG,
    "read_tag": READ_TAG,
    "python": platform.python_version(),
    "numpy": np.__version__,
    "torch": torch.__version__,
    "matplotlib": matplotlib.__version__,
    "scipy": scipy.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
}
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
(LOGS / ("env_" + stamp + ".json")).write_text(json.dumps(ENV, ensure_ascii=False, indent=2))
for k, v in ENV.items():
    print(k.ljust(12), v)


timestamp    2026-10-14T18:57:35
run_tag      w3_artefacts_tong_hop
read_tag     chot
python       3.13.15
numpy        2.1.3
torch        2.11.0+cu130
matplotlib   3.10.0
scipy        1.16.3
gpu          Tesla T4


In [3]:
PALETTE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]

matplotlib.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "legend.frameon": False,
    "lines.linewidth": 1.4,
    "axes.prop_cycle": matplotlib.cycler(color=PALETTE),
    "figure.constrained_layout.use": True,
})


def luu_hinh(fig, ten):
    duong = FIGS / ten
    fig.savefig(duong)
    print("Đã lưu", duong)
    return duong


print("Đã đặt quy ước trình bày hình")


Đã đặt quy ước trình bày hình


In [4]:
# Đợt hợp nhất dùng thêm hai thư mục con để tách bảng và gói nộp.
OUT_PROC = OUT / "processed"
BANG = RUN_DIR / "bang"
GOI = RUN_DIR / "goi"
for d in (BANG, GOI):
    d.mkdir(parents=True, exist_ok=True)


def ghi_bang(ten, cot, hang):
    """Ghi một bảng ra tệp CSV, dùng bảng mã có dấu hiệu thứ tự byte cho Excel."""
    import csv
    duong = BANG / ten
    with open(duong, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f)
        w.writerow(cot)
        for r in hang:
            w.writerow([r.get(k, "") for k in cot])
    print("Đã ghi bảng", ten, "|", len(hang), "dòng |",
          duong.stat().st_size, "byte")
    return duong


def ghi_json(ten, doi_tuong):
    """Ghi một tệp tổng hợp, luôn giữ nguyên dấu tiếng Việt."""
    duong = RUN_DIR / ten
    duong.write_text(json.dumps(doi_tuong, ensure_ascii=False, indent=1),
                     encoding="utf-8")
    print("Đã ghi tệp", ten, "|", duong.stat().st_size, "byte")
    return duong


print("Thư mục bảng:", BANG)
print("Thư mục gói :", GOI)


Thư mục bảng: /content/drive/MyDrive/AIO2026_Conquer_M4_DLinear_NLinear_Slot4/Output/processed/w3_artefacts_tong_hop/bang
Thư mục gói : /content/drive/MyDrive/AIO2026_Conquer_M4_DLinear_NLinear_Slot4/Output/processed/w3_artefacts_tong_hop/goi


In [5]:
# Danh mục mười lần chạy của tuần 3, khai báo tường minh theo ba thành phần:
# Mã sổ tay, tên thư mục con và tên tệp tổng hợp mà sổ tay đó đã ghi ra.
DANH_MUC_W3 = [
    ("W3_16", "w3_16_ba_ham_mat_mat_hai_neo", "W3_ba_ham_mat_mat_hai_neo.json"),
    ("W3_17", "w3_17_luoi_nam_hat_giong", "W3_luoi_nam_hat_giong.json"),
    ("W3_18", "w3_18_chi_so_theo_nhom_kenh", "W3_chi_so_theo_nhom_kenh.json"),
    ("W3_19", "w3_19_phan_tang_theo_do_tre", "W3_phan_tang_theo_do_tre.json"),
    ("W3_20", "w3_20_moc_trung_binh_dich_chuyen",
     "W3_moc_trung_binh_dich_chuyen.json"),
    ("W3_21", "w3_21_doi_chung_moc_manh_nhat", "W3_doi_chung_moc_manh_nhat.json"),
    ("W3_22", "w3_22_neo_b_ham_tuyet_doi_nam_hat_giong",
     "W3_neo_b_mae_nam_hat_giong.json"),
    ("W3_23", "w3_23_kiem_tra_thang_do_kenh", "W3_kiem_tra_thang_do_kenh.json"),
    ("W3_24", "w3_24_neo_theo_nguong_do_phu", "W3_neo_theo_nguong_do_phu.json"),
    ("W3_25", "w3_25_phan_tang_gop_tang_cuoi",
     "W3_phan_tang_gop_tang_cuoi.json"),
]


def doc_tong_hop(ma):
    """Đọc tệp tổng hợp của một sổ tay tuần 3 theo mã sổ tay."""
    for m, thu_muc, ten_tep in DANH_MUC_W3:
        if m == ma:
            duong = OUT / "processed" / thu_muc / ten_tep
            assert duong.exists(), "Không thấy " + str(duong)
            return json.loads(duong.read_text())
    raise KeyError("Không có mã sổ tay " + str(ma))


CO_MAT, THIEU = [], []
for ma, thu_muc, ten_tep in DANH_MUC_W3:
    duong = OUT / "processed" / thu_muc / ten_tep
    (CO_MAT if duong.exists() else THIEU).append(ma)
print("Lần chạy đã có tệp tổng hợp:", ", ".join(CO_MAT))
print("Lần chạy còn thiếu        :", ", ".join(THIEU) if THIEU else "không có")
assert not THIEU, ("Còn thiếu tệp tổng hợp của " + ", ".join(THIEU) +
                   ". Hãy chạy lại các sổ tay đó trước khi hợp nhất.")


Lần chạy đã có tệp tổng hợp: W3_16, W3_17, W3_18, W3_19, W3_20, W3_21, W3_22, W3_23, W3_24, W3_25
Lần chạy còn thiếu        : không có


In [6]:
TEP_HOP_NHAT = RUN_DIR / "W3_HOP_NHAT_CHI_SO.json"
assert TEP_HOP_NHAT.exists(), (
    "Không thấy " + str(TEP_HOP_NHAT) + ". Hãy chạy các sổ tay từ notebook 22.")
HOP_NHAT = json.loads(TEP_HOP_NHAT.read_text())
# Chỉ nhận tệp hợp nhất có phạm vi quét khoá ở tuần 3.
assert HOP_NHAT.get("phien_ban_hop_nhat") == "W3.W4_26", (
    "Tệp hợp nhất chưa được sinh từ notebook 22. Hãy chạy notebook 22 trước.")
print("Phạm vi quét của tệp hợp nhất: tới tuần", HOP_NHAT.get("pham_vi_tuan_toi_da"))
GOC = HOP_NHAT["tong_hop_goc"]
MSE_MOC = HOP_NHAT["mse_persistence"]
MAE_MOC = HOP_NHAT["mae_persistence"]
print("Đã đọc tệp hợp nhất")
print("Số cấu hình:", HOP_NHAT["so_cau_hinh"])
print("Số phép kiểm định:", HOP_NHAT.get("kiem_dinh", {}).get("so_phep", 0))


Phạm vi quét của tệp hợp nhất: tới tuần 3
Đã đọc tệp hợp nhất
Số cấu hình: 18
Số phép kiểm định: 13


## 2. Kiểm tra đầy đủ bộ artefact

Sáu artefact bắt buộc được liệt kê tường minh, củng với yêu cầu đủ sáu hình. Sổ tay dừng lại nếu thiếu bất cứ mục nào. Một gói artefact thiếu thành phần tệ hơn không có gói nào, vì người dùng sẽ tưởng mình đã có đầy đủ rồi quay lại trích từ nguồn cũ.

In [7]:
BAT_BUOC = [
    (RUN_DIR / "W3_HOP_NHAT_CHI_SO.json", "tệp hợp nhất mọi số liệu"),
    (BANG / "W3_bang_tung_lan_huan_luyen.csv", "bảng theo từng lần huấn luyện"),
    (BANG / "W3_bang_theo_cau_hinh.csv", "bảng theo từng cấu hình"),
    (BANG / "W3_bang_kiem_dinh_hop_nhat.csv", "bảng kiểm định đồng nhất"),
    (BANG / "W3_bang_kenh_hop_nhat.csv", "bảng theo kênh"),
    (BANG / "W3_bang_phan_tang_hop_nhat.csv", "bảng phân tầng theo độ trễ"),
]
thieu = []
for duong, mo_ta in BAT_BUOC:
    if duong.exists():
        print("có   ", duong.name.ljust(38),
              str(duong.stat().st_size).rjust(8), "byte |", mo_ta)
    else:
        thieu.append(duong.name)
        print("thiếu", duong.name.ljust(38), "|", mo_ta)
HINH = sorted(FIGS.glob("W3_T*.png"))
print("")
print("Số hình của đợt hợp nhất:", len(HINH))
for p in HINH:
    print("   ", p.name.ljust(44), p.stat().st_size, "byte")
assert not thieu, ("Thiếu artefact: " + ", ".join(thieu) +
                   ". Hãy chạy lại các sổ tay từ notebook 22.")
assert len(HINH) >= 6, "Chưa đủ sáu hình, hãy chạy lại notebook 24 và notebook 25"
print("")
print("Bộ artefact đầy đủ")


có    W3_HOP_NHAT_CHI_SO.json                  225847 byte | tệp hợp nhất mọi số liệu
có    W3_bang_tung_lan_huan_luyen.csv           14581 byte | bảng theo từng lần huấn luyện
có    W3_bang_theo_cau_hinh.csv                  3590 byte | bảng theo từng cấu hình
có    W3_bang_kiem_dinh_hop_nhat.csv             4743 byte | bảng kiểm định đồng nhất
có    W3_bang_kenh_hop_nhat.csv                  9005 byte | bảng theo kênh
có    W3_bang_phan_tang_hop_nhat.csv             3089 byte | bảng phân tầng theo độ trễ

Số hình của đợt hợp nhất: 6
    W3_T01_thu_hang_cau_hinh.png                 253083 byte
    W3_T02_khoang_tin_cay_hop_nhat.png           262186 byte
    W3_T03_hieu_ung_theo_kenh.png                166317 byte
    W3_T04_phan_tang_do_tre.png                  167997 byte
    W3_T05_ho_moc_va_hai_neo.png                 173485 byte
    W3_T06_on_dinh_va_ket_qua_am_tinh.png        254399 byte

Bộ artefact đầy đủ


## 3. Sinh bản trích số liệu

Bản trích được tổ chức theo từng mục của báo cáo, mỗi dòng một con số kèm nhãn tiếng Việt và kèm chỉ dẫn nguồn. Chỉ dẫn nguồn là thành phần bắt buộc: nó cho phép bất cứ ai đọc báo cáo cũng truy nguyên một con số về đúng lần chạy đã sinh ra nó, không cần hỏi lại tác giả.

In [8]:
def so(v, n=6):
    if v is None:
        return "chưa có"
    return str(round(float(v), n))


dong = []
dong.append("BẢN TRÍCH SỐ LIỆU TUẦN 3")
dong.append("Sinh tự động từ W3_HOP_NHAT_CHI_SO.json, không chép tay")
dong.append("Thống kê lúc: " +
            datetime.datetime.now().isoformat(timespec="seconds"))
dong.append("")
dong.append("1. MỐC THAM CHIẾU VÀ MÔ HÌNH CƠ SỞ PERSISTENCE")
dong.append("   Sai số bình phương của mốc giữ nguyên dự báo cuối: " +
            so(MSE_MOC, 10))
dong.append("   Sai số tuyệt đối của mốc: " + so(MAE_MOC, 10))
dong.append("   Mọi lần tính lại mốc trùng khít: " +
            str(HOP_NHAT["lien_thong_moc"]))
dong.append("   Số lần huấn luyện trong sổ đăng ký: " +
            str(HOP_NHAT["so_lan_huan_luyen"]))
dong.append("   Số cấu hình phân biệt: " + str(HOP_NHAT["so_cau_hinh"]))
dong.append("   Số lần được dùng sau khi gộp bản sao: " +
            str(HOP_NHAT.get("so_lan_duoc_dung")))
dong.append("   Số cặp bản sao lệch nhau quá ngưỡng: " +
            str(len(HOP_NHAT.get("ban_sao_lech_nhau", []))))
dong.append("   Nguồn: notebook 22")
dong.append("")
dong.append("2. CẤU HÌNH XẾP THEO CHỈ TIÊU CHÍNH")
for r in HOP_NHAT["cau_hinh"]:
    dong.append("   " + r["khoa_cau_hinh"] + ": bình phương " +
                so(r["mse_trung_binh"]) + " cộng trừ " +
                so(r["mse_do_lech_chuan"]) + "; tuyệt đối " +
                so(r["mae_trung_binh"]) + " cộng trừ " +
                so(r["mae_do_lech_chuan"]) + "; năng lực " +
                so(r["nang_luc_theo_mae"]) + "; hạt giống " +
                str(r["so_hat_giong"]) + "; nguồn " + r["nguon"])
dong.append("   Tốt nhất theo chỉ tiêu chính: " +
            str(HOP_NHAT["tot_nhat_theo_mse"]))
dong.append("   Tốt nhất theo chỉ tiêu phụ: " +
            str(HOP_NHAT["tot_nhat_theo_mae"]))
dong.append("")
dong.append("3. TỶ SỐ BIẾN THIÊN THEO TỪNG CHỈ TIÊU")
for r in HOP_NHAT.get("ty_so_bien_thien", []):
    dong.append("   " + r["khoa_cau_hinh"] + ": tỷ số trên bình phương " +
                so(r["ty_so_mse"], 4) + "; tỷ số trên tuyệt đối " +
                so(r["ty_so_mae"], 4))
dong.append("   Mọi phát biểu về biến thiên phải gắn tên chỉ tiêu")
dong.append("")
print("Đã dựng phần đầu, hiện có", len(dong), "dòng")


Đã dựng phần đầu, hiện có 50 dòng


In [9]:
kd = HOP_NHAT.get("kiem_dinh", {})
dong.append("4. KẾT QUẢ KIỂM ĐỊNH ĐỒNG NHẤT")
dong.append("   Số lần lấy mẫu lặp: " + str(kd.get("n_boot")) +
            "; hạt giống lấy mẫu: " + str(kd.get("hat_giong_lay_mau")) +
            "; đơn vị lấy mẫu: " + str(kd.get("don_vi_lay_mau")))
for r in kd.get("ket_qua", []):
    dong.append("   " + r["ma"] + " " + r["ben_trai"] + " trừ " + r["ben_phai"])
    dong.append("      câu hỏi: " + r["cau_hoi"])
    dong.append("      bình phương " + so(r["hieu_mse"]) + " khoảng từ " +
                so(r["hieu_mse_ci"][0]) + " đến " + so(r["hieu_mse_ci"][1]) +
                "; giá trị p " + so(r["p_hai_phia_mse"], 4) + "; " +
                r["ket_luan_mse"])
    dong.append("      tuyệt đối " + so(r["hieu_mae"]) + " khoảng từ " +
                so(r["hieu_mae_ci"][0]) + " đến " + so(r["hieu_mae_ci"][1]) +
                "; giá trị p " + so(r["p_hai_phia_mae"], 4) + "; " +
                r["ket_luan_mae"])
dong.append("   Nguồn: notebook 23")
dong.append("")
tg = HOP_NHAT.get("truc_giai_thich", {})
dong.append("5. TRỤC KÊNH")
dong.append("   Số kênh: " + str(tg.get("so_kenh")))
dong.append("   Tương quan độ phủ và hiệu ứng: " +
            so(tg.get("tuong_quan_do_phu_va_hieu_ung")))
dong.append("   Kênh bị gắn cờ thang đo: " +
            (", ".join(tg.get("kenh_co_co_cong_bo", [])) or "không có"))
dong.append("   Kênh được gỡ cờ so với bản sàng lọc: " +
            (", ".join(tg.get("kenh_duoc_go_co", [])) or "không có"))
dong.append("   Tương quan độ lệch chuẩn và sai số neo bước cuối, toàn bộ: " +
            so(tg.get("tuong_quan_do_lech_va_mse_neoA_toan_bo")))
dong.append("   Cùng hệ số sau khi loại kênh có cờ: " +
            so(tg.get("tuong_quan_do_lech_va_mse_neoA_sau_loai")))
dong.append("   Hệ số này không ổn định nên không trích làm bằng chứng")
dong.append("   Nguồn: notebook 24")
dong.append("")
dong.append("6. TRỤC ĐỘ TRỄ")
dong.append("   Cách chia chính thức: " + str(tg.get("tang_chinh_thuc")))
for r in tg.get("bang_phan_tang", []):
    if not r.get("la_dong_chinh_thuc"):
        continue
    dong.append("   " + r["tang"] + ": " + str(r["so_diem"]) + " điểm, chiếm " +
                so(100.0 * float(r["ty_le_diem"]), 3) + " phần trăm; hiệu " +
                so(r["hieu_mse"]) + "; giá trị p " + so(r["p_mse"], 4) +
                "; đóng góp " +
                so(100.0 * float(r["dong_gop_vao_hieu_ung_chung"]), 3) +
                " phần trăm")
dong.append("   Nguồn: notebook 24")
dong.append("")
dl = HOP_NHAT.get("doi_chieu_neo_lai", {})
dong.append("7. KẾT QUẢ ÂM TÍNH CỦA LUẬT CHỌN NEO")
dong.append("   Kênh bị chuyển về neo bước cuối: " +
            (", ".join(dl.get("kenh_bi_chuyen", [])) or "không có"))
dong.append("   Kênh mà neo quan sát thật thực sự làm xấu: " +
            (", ".join(dl.get("kenh_neo_lam_xau", [])) or "không có"))
dong.append("   Kênh trùng giữa hai tập: " +
            (", ".join(dl.get("kenh_trung", [])) or "không có"))
dong.append("   Tổng lợi ích bị trả lại: " + so(dl.get("loi_ich_bi_tra_lai")))
dong.append("   Nguồn: notebook 25")
dong.append("")
dong.append("8. DANH MỤC HÌNH")
for ten, mo_ta in sorted(HOP_NHAT.get("danh_muc_hinh", {}).items()):
    dong.append("   " + ten + ": " + mo_ta)
dong.append("")
BAN_TRICH = BANG / "W3_ban_trich_so_lieu.txt"
BAN_TRICH.write_text("\n".join(dong), encoding="utf-8")
print("Đã ghi bản trích:", BAN_TRICH)
print("Số dòng:", len(dong), "| kích cỡ:", BAN_TRICH.stat().st_size, "byte")
print("")
print("\n".join(dong[:40]))


Đã ghi bản trích: /content/drive/MyDrive/AIO2026_Conquer_M4_DLinear_NLinear_Slot4/Output/processed/w3_artefacts_tong_hop/bang/W3_ban_trich_so_lieu.txt
Số dòng: 141 | kích cỡ: 12775 byte

BẢN TRÍCH SỐ LIỆU TUẦN 3
Sinh tự động từ W3_HOP_NHAT_CHI_SO.json, không chép tay
Thống kê lúc: 2026-10-14T18:57:36

1. MỐC THAM CHIẾU VÀ MÔ HÌNH CƠ SỞ PERSISTENCE
   Sai số bình phương của mốc giữ nguyên dự báo cuối: 0.5787542077
   Sai số tuyệt đối của mốc: 0.4659432815
   Mọi lần tính lại mốc trùng khít: True
   Số lần huấn luyện trong sổ đăng ký: 101
   Số cấu hình phân biệt: 18
   Số lần được dùng sau khi gộp bản sao: 76
   Số cặp bản sao lệch nhau quá ngưỡng: 0
   Nguồn: notebook 22

2. CẤU HÌNH XẾP THEO CHỈ TIÊU CHÍNH
   nlinear|B|huber: bình phương 0.433972 cộng trừ 0.000291; tuyệt đối 0.43127 cộng trừ 0.000668; năng lực 0.074414; hạt giống 5; nguồn bo_sung_14_huber_neo_b,w3_17_luoi_nam_hat_giong
   dlinear|B|huber: bình phương 0.434091 cộng trừ 0.000211; tuyệt đối 0.431355 cộng trừ 0.000713; nă

## 4. Danh mục tệp kèm mã băm

Mã băm được tính bằng thuật toán số học mặc định của thư viện chuẩn, không cần cài thêm gói nào. Mục đích không phải bảo mật mà là định danh phiên bản: khi bản báo cáo đã nộp và sau đó có người chạy lại thực nghiệm, hai bên đối chiếu mã băm là biết ngay đang nói về cùng một bộ số liệu hay không.

In [10]:
import hashlib

def bam(duong):
    h = hashlib.sha256()
    with open(duong, "rb") as f:
        for khoi in iter(lambda: f.read(1 << 16), b""):
            h.update(khoi)
    return h.hexdigest()


DANH_MUC = []
nguon = ([RUN_DIR / "W3_HOP_NHAT_CHI_SO.json"] +
         sorted(BANG.glob("*.csv")) + [BANG / "W3_ban_trich_so_lieu.txt"] +
         sorted(FIGS.glob("W3_T*.png")))
for p in nguon:
    if not p.exists():
        continue
    DANH_MUC.append({"ten_tep": p.name,
                     "duong_dan_tuong_doi": str(p.relative_to(OUT)),
                     "kich_co_byte": p.stat().st_size,
                     "ma_bam_sha256": bam(p)})
ghi_bang("W3_danh_muc_artefact.csv",
         ["ten_tep", "duong_dan_tuong_doi", "kich_co_byte", "ma_bam_sha256"],
         DANH_MUC)
for r in DANH_MUC:
    print(r["ten_tep"].ljust(44), str(r["kich_co_byte"]).rjust(9), "byte",
          r["ma_bam_sha256"][:16])
print("")
print("Tổng số tệp trong gói:", len(DANH_MUC))
print("Tổng kích cỡ:", sum(r["kich_co_byte"] for r in DANH_MUC), "byte")


Đã ghi bảng W3_danh_muc_artefact.csv | 14 dòng | 2421 byte
W3_HOP_NHAT_CHI_SO.json                         225847 byte 82c2937ef983d221
W3_bang_giao_thuc_khac.csv                         155 byte bcf0c3dc61f663b2
W3_bang_kenh_hop_nhat.csv                         9005 byte 073d42c6022a0406
W3_bang_kiem_dinh_hop_nhat.csv                    4743 byte 76970ce00f43d239
W3_bang_phan_tang_hop_nhat.csv                    3089 byte af873464fcba37c6
W3_bang_theo_cau_hinh.csv                         3590 byte 5f2f99518bbbe5e2
W3_bang_tung_lan_huan_luyen.csv                  14581 byte 988f350dde32d4b4
W3_ban_trich_so_lieu.txt                         12775 byte fa256ce11e07ce1e
W3_T01_thu_hang_cau_hinh.png                    253083 byte f5c72ee8abcfe363
W3_T02_khoang_tin_cay_hop_nhat.png              262186 byte f6f88425c8664e6b
W3_T03_hieu_ung_theo_kenh.png                   166317 byte ce01b05f1dd409b7
W3_T04_phan_tang_do_tre.png                     167997 byte cfc0825c8b9ae254
W3_T05_ho_moc_va_

## 5. Nén gói và tải về

Gói được nén vào chính thư mục của lần chạy này, nên nó nằm trên ổ đĩa chia sẻ và không mất khi phiên làm việc kết thúc. Bên trong gói, các tệp được xếp theo đúng đường dẫn tương đối của chúng, nên giải nén ra là có ngay cấu trúc cần cho bản soạn thảo.

In [11]:
import shutil
import zipfile

TEP_NEN = RUN_DIR / (TEN_GOI + ".zip")
if TEP_NEN.exists():
    TEP_NEN.unlink()
with zipfile.ZipFile(TEP_NEN, "w", zipfile.ZIP_DEFLATED) as z:
    for r in DANH_MUC:
        z.write(OUT / r["duong_dan_tuong_doi"],
                arcname=TEN_GOI + "/" + r["ten_tep"])
    z.write(BANG / "W3_danh_muc_artefact.csv",
            arcname=TEN_GOI + "/W3_danh_muc_artefact.csv")
print("Đã nén:", TEP_NEN)
print("Kích cỡ gói:", TEP_NEN.stat().st_size, "byte")
with zipfile.ZipFile(TEP_NEN) as z:
    for ten in z.namelist():
        print("   ", ten)
if TAI_VE_SAU_KHI_NEN:
    try:
        from google.colab import files as colab_files
        colab_files.download(str(TEP_NEN))
        print("Đã gọi hộp thoại tải về")
    except Exception as loi:
        print("Không tải về tự động được:", loi)
        print("Hãy lấy gói trực tiếp từ đường dẫn trên ổ đĩa chia sẻ")


Đã nén: /content/drive/MyDrive/AIO2026_Conquer_M4_DLinear_NLinear_Slot4/Output/processed/w3_artefacts_tong_hop/W3_artefacts_bao_cao.zip
Kích cỡ gói: 1072943 byte
    W3_artefacts_bao_cao/W3_HOP_NHAT_CHI_SO.json
    W3_artefacts_bao_cao/W3_bang_giao_thuc_khac.csv
    W3_artefacts_bao_cao/W3_bang_kenh_hop_nhat.csv
    W3_artefacts_bao_cao/W3_bang_kiem_dinh_hop_nhat.csv
    W3_artefacts_bao_cao/W3_bang_phan_tang_hop_nhat.csv
    W3_artefacts_bao_cao/W3_bang_theo_cau_hinh.csv
    W3_artefacts_bao_cao/W3_bang_tung_lan_huan_luyen.csv
    W3_artefacts_bao_cao/W3_ban_trich_so_lieu.txt
    W3_artefacts_bao_cao/W3_T01_thu_hang_cau_hinh.png
    W3_artefacts_bao_cao/W3_T02_khoang_tin_cay_hop_nhat.png
    W3_artefacts_bao_cao/W3_T03_hieu_ung_theo_kenh.png
    W3_artefacts_bao_cao/W3_T04_phan_tang_do_tre.png
    W3_artefacts_bao_cao/W3_T05_ho_moc_va_hai_neo.png
    W3_artefacts_bao_cao/W3_T06_on_dinh_va_ket_qua_am_tinh.png
    W3_artefacts_bao_cao/W3_danh_muc_artefact.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Đã gọi hộp thoại tải về


## 6. Kết luận của đợt hợp nhất

Năm sổ tay của nhánh này không tạo thêm bất cứ số liệu thực nghiệm nào. Chúng đọc lại, đối chiếu, kiểm tra tính liên thông, hợp nhất và đóng gói. Đó chính là điều làm cho chúng an toàn: một sổ tay không huấn luyện thì không thể làm thay đổi kết quả, chỉ có thể làm lộ ra mâu thuẫn đang ẩn.

Tất cả ghi vào một thư mục duy nhất. Các thư mục của mười lần chạy gốc không bị chỉnh sửa, nên luôn có thể dựng lại toàn bộ gói từ đầu nếu cần.

In [12]:
print("TÓM LƯỢC NHÁNH CHỮ B")
print("")
print("Thư mục hợp nhất :", RUN_DIR)
print("Số bảng dạng CSV:", len(sorted(BANG.glob("*.csv"))))
print("Số hình        :", len(sorted(FIGS.glob("W3_T*.png"))))
print("Số tệp trong gói:", len(DANH_MUC))
print("Gói nộp        :", TEP_NEN.name)
print("")
print("Ba quy tắc chốt lại cho báo cáo tuần 3")
print("   một, chỉ tiêu chính là sai số bình phương trên các điểm có phép đo thật")
print("   hai, mọi con số trích từ bản trích số liệu, không chép từ màn hình")
print("   ba, mọi phát biểu về biến thiên phải gắn tên chỉ tiêu")


TÓM LƯỢC NHÁNH CHỮ B

Thư mục hợp nhất : /content/drive/MyDrive/AIO2026_Conquer_M4_DLinear_NLinear_Slot4/Output/processed/w3_artefacts_tong_hop
Số bảng dạng CSV: 7
Số hình        : 6
Số tệp trong gói: 14
Gói nộp        : W3_artefacts_bao_cao.zip

Ba quy tắc chốt lại cho báo cáo tuần 3
   một, chỉ tiêu chính là sai số bình phương trên các điểm có phép đo thật
   hai, mọi con số trích từ bản trích số liệu, không chép từ màn hình
   ba, mọi phát biểu về biến thiên phải gắn tên chỉ tiêu
